In [1]:
import sqlite3

# Tworzy plik "wypozyczalnia.db" w Twoim folderze (lub łączy się z nim, jeśli istnieje)
conn_sqlite = sqlite3.connect('wypozyczalnia.db')
cursor_sqlite = conn_sqlite.cursor()

# Tu wklejasz kod z modelu fizycznego SQLite
cursor_sqlite.executescript("""
CREATE TABLE IF NOT EXISTS Kategorie (
    ID_Kategorii INTEGER PRIMARY KEY,
    Nazwa TEXT NOT NULL,
    Cena_Za_Dzien REAL NOT NULL
);

-- i reszta Twoich tabel...
""")

conn_sqlite.commit()
print("Baza SQLite utworzona pomyślnie!")

cursor_sqlite.close()
conn_sqlite.close()

Baza SQLite utworzona pomyślnie!


In [3]:
import sqlite3

# Łączymy się z utworzonym wcześniej plikiem
conn_sqlite = sqlite3.connect('wypozyczalnia.db')
cursor_sqlite = conn_sqlite.cursor()

# 1. CREATE i INSERT w jednym dużym skrypcie
cursor_sqlite.executescript("""
-- ============================
-- TWORZENIE TABEL (DDL) - WERSJA SQLITE
-- ============================

CREATE TABLE IF NOT EXISTS Kategorie (
    ID_Kategorii INTEGER PRIMARY KEY AUTOINCREMENT,
    Nazwa TEXT NOT NULL,
    Cena_Za_Dzien REAL NOT NULL
);

CREATE TABLE IF NOT EXISTS Klienci (
    ID_Klienta INTEGER PRIMARY KEY AUTOINCREMENT,
    Imie TEXT NOT NULL,
    Nazwisko TEXT NOT NULL,
    PESEL TEXT UNIQUE NOT NULL,
    Nr_Prawa_Jazdy TEXT UNIQUE,
    Telefon TEXT,
    Email TEXT
);

CREATE TABLE IF NOT EXISTS Samochody (
    ID_Samochodu INTEGER PRIMARY KEY AUTOINCREMENT,
    Marka TEXT NOT NULL,
    Model TEXT NOT NULL,
    Nr_Rejestracyjny TEXT UNIQUE NOT NULL,
    Rok_Produkcji INTEGER,
    ID_Kategorii INTEGER,
    FOREIGN KEY (ID_Kategorii) REFERENCES Kategorie(ID_Kategorii)
);

-- Tabela pracownicy musi powstać przed Wypozyczeniami!
CREATE TABLE IF NOT EXISTS pracownicy (
    id_pracownika INTEGER PRIMARY KEY AUTOINCREMENT,
    imie TEXT NOT NULL,
    nazwisko TEXT NOT NULL,
    stanowisko TEXT NOT NULL,
    telefon TEXT UNIQUE
);

CREATE TABLE IF NOT EXISTS Wypozyczenia (
    ID_Wypozyczenia INTEGER PRIMARY KEY AUTOINCREMENT,
    ID_Klienta INTEGER,
    ID_Samochodu INTEGER,
    id_pracownika INTEGER,
    Data_Od TEXT NOT NULL,
    Data_Do TEXT NOT NULL,
    Status TEXT NOT NULL,
    FOREIGN KEY (ID_Klienta) REFERENCES Klienci(ID_Klienta),
    FOREIGN KEY (ID_Samochodu) REFERENCES Samochody(ID_Samochodu),
    FOREIGN KEY (id_pracownika) REFERENCES pracownicy(id_pracownika)
);

CREATE TABLE IF NOT EXISTS platnosci (
    id_platnosci INTEGER PRIMARY KEY AUTOINCREMENT,
    id_wypozyczenia INTEGER NOT NULL,
    kwota REAL NOT NULL CHECK (kwota > 0),
    data_platnosci TEXT NOT NULL DEFAULT CURRENT_DATE,
    metoda_platnosci TEXT NOT NULL CHECK (metoda_platnosci IN ('Karta', 'Gotówka', 'Przelew', 'BLIK')),
    FOREIGN KEY (id_wypozyczenia) REFERENCES Wypozyczenia(ID_Wypozyczenia)
);

-- ============================
-- WSTAWIANIE DANYCH (DML)
-- ============================

INSERT INTO Kategorie (Nazwa, Cena_Za_Dzien) VALUES 
('Kompakt', 150.00),
('SUV', 250.00),
('Premium', 400.00);

INSERT INTO Klienci (Imie, Nazwisko, PESEL, Nr_Prawa_Jazdy, Telefon, Email) VALUES 
('Jan', 'Kowalski', '90051412345', '12345/67/8901', '+48123456789', 'jan.kowalski@email.com'),
('Anna', 'Nowak', '85112298765', '98765/43/2109', '+48987654321', 'anna.nowak@email.com');

INSERT INTO Samochody (Marka, Model, Nr_Rejestracyjny, Rok_Produkcji, ID_Kategorii) VALUES 
('Toyota', 'Corolla', 'WA12345', 2022, 1),
('Kia', 'Sportage', 'KR54321', 2023, 2),
('BMW', 'Seria 5', 'PO98765', 2024, 3);

INSERT INTO pracownicy (imie, nazwisko, stanowisko, telefon) VALUES
('Marek', 'Kowalski', 'Kierownik', '111222333');

-- Przypisujemy pracownika o ID=1 do każdego wypożyczenia testowego
INSERT INTO Wypozyczenia (ID_Klienta, ID_Samochodu, id_pracownika, Data_Od, Data_Do, Status) VALUES 
(1, 1, 1, '2026-06-01', '2026-06-05', 'Zakończone'),
(2, 2, 1, '2026-06-10', '2026-06-15', 'W_trakcie'),
(1, 3, 1, '2026-07-01', '2026-07-03', 'Zarezerwowane');

INSERT INTO platnosci (id_wypozyczenia, kwota, data_platnosci, metoda_platnosci) VALUES
(1, 450.00, '2026-06-05', 'Karta');
""")

# Pamiętaj o zapisaniu zmian!
conn_sqlite.commit()
print("Baza SQLite uzupełniona strukturą i danymi!")

# 2. Szybki test czy działa (zapytanie SELECT)
cursor_sqlite.execute("SELECT Marka, Model, Nr_Rejestracyjny FROM Samochody;")
test_samochody = cursor_sqlite.fetchall()

print("\nPobrane auta z SQLite:")
for auto in test_samochody:
    print(f"- {auto[0]} {auto[1]} (Rej: {auto[2]})")

# 3. Zamykamy
cursor_sqlite.close()
conn_sqlite.close()

Baza SQLite uzupełniona strukturą i danymi!

Pobrane auta z SQLite:
- Toyota Corolla (Rej: WA12345)
- Kia Sportage (Rej: KR54321)
- BMW Seria 5 (Rej: PO98765)


In [6]:
conn_sqlite = sqlite3.connect('wypozyczalnia.db')
cursor_sqlite = conn_sqlite.cursor()
cursor_sqlite.execute("SELECT Marka, Model, Nr_Rejestracyjny FROM Samochody;")
test_samochody = cursor_sqlite.fetchall()
print("\nPobrane auta z SQLite:")
for auto in test_samochody:
    print(f"- {auto[0]} {auto[1]} (Rej: {auto[2]})")
cursor_sqlite.close()
conn_sqlite.close()


Pobrane auta z SQLite:
- Toyota Corolla (Rej: WA12345)
- Kia Sportage (Rej: KR54321)
- BMW Seria 5 (Rej: PO98765)


In [1]:
import csv
import random

# Przykładowe dane do wygenerowania większego zbioru
imiona = ["Jan", "Anna", "Piotr", "Katarzyna", "Tomasz", "Magdalena", "Marcin", "Agnieszka", "Krzysztof", "Ewa"]
nazwiska = ["Kowalski", "Nowak", "Wiśniewski", "Wójcik", "Kowalczyk", "Kamiński", "Lewandowski", "Zieliński", "Szymański", "Woźniak"]

dane_do_csv = []
for i in range(10):
    imie = random.choice(imiona)
    nazwisko = random.choice(nazwiska)
    pesel = f"{random.randint(50, 99):02d}{random.randint(1, 12):02d}{random.randint(1, 28):02d}{random.randint(10000, 99999)}"
    nr_pj = f"{random.randint(10000, 99999)}/{random.randint(10, 99)}/{random.randint(1000, 9999)}"
    telefon = f"+48{random.randint(100000000, 999999999)}"
    email = f"{imie.lower()}.{nazwisko.lower()}{random.randint(1,100)}@email.com"
    
    dane_do_csv.append([imie, nazwisko, pesel, nr_pj, telefon, email])

# Zapis do pliku CSV
with open('klienci.csv', 'w', newline='', encoding='utf-8') as plik_csv:
    writer = csv.writer(plik_csv)
    # Zapisujemy nagłówki
    writer.writerow(['Imie', 'Nazwisko', 'PESEL', 'Nr_Prawa_Jazdy', 'Telefon', 'Email'])
    # Zapisujemy wygenerowane wiersze
    writer.writerows(dane_do_csv)

print("Plik klienci.csv został wygenerowany!")

Plik klienci.csv został wygenerowany!


In [2]:
import csv
import psycopg
import simplejson
import sqlite3

# 1. Odczyt przygotowanych danych z pliku CSV (pomijamy pierwszy wiersz z nagłówkami)
dane_z_csv = []
with open('klienci.csv', 'r', encoding='utf-8') as plik_csv:
    reader = csv.reader(plik_csv)
    next(reader) # pomija nagłówek
    for row in reader:
        dane_z_csv.append(row)

print(f"Pobrano {len(dane_z_csv)} rekordów z pliku CSV. Rozpoczynamy import...")

# ==========================================
# IMPORT DO POSTGRESQL
# ==========================================
try:
    with open("database_creds.json") as db_con_file:
        creds = simplejson.loads(db_con_file.read())
        
    conn_pg = psycopg.connect(
        host=creds['host_name'], user=creds['user_name'],
        dbname=creds['db_name'], password=creds['password'], port=creds['port_number']
    )
    cursor_pg = conn_pg.cursor()
    
    # Używamy executemany dla batch insertu (z %s dla psycopg)
    zapytanie_pg = """
        INSERT INTO Klienci (Imie, Nazwisko, PESEL, Nr_Prawa_Jazdy, Telefon, Email) 
        VALUES (%s, %s, %s, %s, %s, %s)
    """
    cursor_pg.executemany(zapytanie_pg, dane_z_csv)
    conn_pg.commit()
    
    print("✅ Pomyślnie zaimportowano do PostgreSQL!")
    cursor_pg.close()
    conn_pg.close()
except Exception as e:
    print("❌ Błąd PostgreSQL:", e)

# ==========================================
# IMPORT DO SQLITE
# ==========================================
try:
    conn_sl = sqlite3.connect('wypozyczalnia.db')
    cursor_sl = conn_sl.cursor()
    
    # Używamy executemany dla batch insertu (z ? dla sqlite3)
    zapytanie_sl = """
        INSERT INTO Klienci (Imie, Nazwisko, PESEL, Nr_Prawa_Jazdy, Telefon, Email) 
        VALUES (?, ?, ?, ?, ?, ?)
    """
    cursor_sl.executemany(zapytanie_sl, dane_z_csv)
    conn_sl.commit()
    
    print("✅ Pomyślnie zaimportowano do SQLite!")
    cursor_sl.close()
    conn_sl.close()
except Exception as e:
    print("❌ Błąd SQLite:", e)

Pobrano 10 rekordów z pliku CSV. Rozpoczynamy import...
✅ Pomyślnie zaimportowano do PostgreSQL!
✅ Pomyślnie zaimportowano do SQLite!


In [ ]:
S